In [0]:
%sql
CREATE OR REPLACE TABLE data_warehouse_factory.silver.silver_employees AS
WITH cleaned_source AS (
  SELECT 
    trim(employee_full_name) AS employee_full_name,
    element_at(split(trim(employee_full_name), ' '), 1) AS first_name,
    element_at(split(trim(employee_full_name), ' '), -1) AS last_name,
    upper(trim(work_center)) AS work_center,
    CAST(date_of_employment AS DATE) AS date_of_employment,
    CAST(date_of_leaving AS DATE) AS date_of_leaving,
    _source_file,
    _bronze_ingested_at
  FROM data_warehouse_factory.bronze.bronze_employees
  WHERE employee_full_name IS NOT NULL AND trim(employee_full_name) != ''
),

-- 1. Deduplikacja: wybór najświeższego rekordu per pracownik
deduplicated_employees AS (
  SELECT 
    *,
    ROW_NUMBER() OVER (
      PARTITION BY employee_full_name 
      ORDER BY _bronze_ingested_at DESC
    ) AS rnk
  FROM cleaned_source
),

-- 2. Nadanie stabilnego identyfikatora i wyliczenie metryk pracowniczych
final_silver AS (
  SELECT 
    CAST(ROW_NUMBER() OVER (ORDER BY employee_full_name ASC) AS INT) AS employee_key,
    employee_full_name,
    first_name,
    last_name,
    work_center,
    date_of_employment,
    date_of_leaving,

    CASE 
      WHEN date_of_employment IS NOT NULL 
       AND (date_of_leaving IS NULL OR date_of_leaving >= CURRENT_DATE()) 
      THEN TRUE 
      ELSE FALSE 
    END AS is_active,

    timestampdiff(
      MONTH, 
      date_of_employment, 
      COALESCE(date_of_leaving, CURRENT_DATE())
    ) AS tenure_months,

    _source_file,
    _bronze_ingested_at,
    current_timestamp() AS _silver_ingested_at
  FROM deduplicated_employees
  WHERE rnk = 1
)

SELECT * FROM final_silver;